In [14]:
import duckdb
con = duckdb.connect("local.duckdb")

In [15]:
# Merge 1: Party + PartyRelationship → Identity_All
con.execute("""
CREATE OR REPLACE TABLE Identity_All AS
WITH
p AS (SELECT * FROM read_csv_auto('Party.csv', header=True)),
pr AS (SELECT * FROM read_csv_auto('PartyRelationship.csv', header=True))
SELECT
  p.*,

  pr.PartyRelationshipId AS pr_PartyRelationshipId,
  pr.RelatedPartyId      AS pr_RelatedPartyId,
  pr.Roles               AS pr_Roles,
  pr.* EXCLUDE (PartyRelationshipId, RelatedPartyId, Roles, PartyId)
FROM p
LEFT JOIN pr
  ON p.PartyId = pr.PartyId
""")

In [16]:
# Merge 2: BroadridgeAccount + BroadridgeAccountRole + Party → Broadridge_All
con.execute("""
CREATE OR REPLACE TABLE Broadridge_Core_All AS
WITH
ba  AS (SELECT * FROM read_csv_auto('BroadridgeAccount.csv', header=True)),
bar AS (SELECT * FROM read_csv_auto('BroadridgeAccountRole.csv', header=True)),
p   AS (SELECT * FROM read_csv_auto('Party.csv', header=True))
SELECT
  ba.*,

  bar.FinancialAccountRoleId AS bar_FinancialAccountRoleId,
  bar.AccountId              AS bar_AccountId,
  bar.PartyId                AS bar_PartyId,
  bar.FinancialAccountRole   AS bar_FinancialAccountRole,
  bar.* EXCLUDE (FinancialAccountRoleId, AccountId, PartyId, FinancialAccountRole),

  -- Party attributes for role party (prefix)
  p.PartyId AS rp_PartyId,
  p.* EXCLUDE (PartyId)
FROM ba
LEFT JOIN bar
  ON ba.FinancialAccountId = bar.AccountId
LEFT JOIN p
  ON bar.PartyId = p.PartyId
""")

In [17]:
# Merge 3：ContributionHistory + DistributionHistory → Broadridge_Activity_All
con.execute("""
CREATE OR REPLACE TABLE Broadridge_Activity_All AS
WITH
c AS (SELECT * FROM read_csv_auto('BroadridgeAccountContributionHistory.csv', header=True)),
d AS (SELECT * FROM read_csv_auto('BroadridgeAccountDistributionHistory.csv', header=True))
SELECT
  COALESCE(c.FinancialAccountId, d.FinancialAccountId) AS FinancialAccountId,
  COALESCE(c.DeltaDate, d.DeltaDate)                   AS DeltaDate,
  c.* EXCLUDE (FinancialAccountId, DeltaDate),
  d.* EXCLUDE (FinancialAccountId, DeltaDate)
FROM c
FULL OUTER JOIN d
  ON c.FinancialAccountId = d.FinancialAccountId
 AND c.DeltaDate = d.DeltaDate
""")

In [18]:
# Merge 4: CALAccount + BroadridgeAccountToCALAssociation
con.execute("""
CREATE OR REPLACE TABLE CAL_All AS
WITH
cal AS (SELECT * FROM read_csv_auto('CALAccount.csv', header=True)),
assoc AS (SELECT * FROM read_csv_auto('BroadridgeAccountToCALAssociation.csv', header=True)),
p AS (SELECT * FROM read_csv_auto('Party.csv', header=True))
SELECT
  assoc.BroadridgeFinancialAccountId AS cal_BroadridgeFinancialAccountId,
  assoc.CALFinancialAccountNumber    AS cal_CALFinancialAccountNumber,
  assoc.* EXCLUDE (BroadridgeFinancialAccountId, CALFinancialAccountNumber),

  cal.PrimaryOwnerPartyId            AS cal_PrimaryOwnerPartyId,
  cal.FinancialAccountNumber         AS cal_FinancialAccountNumber,
  cal.* EXCLUDE (PrimaryOwnerPartyId, FinancialAccountNumber),

  p.PartyId AS calOwner_PartyId,
  p.* EXCLUDE (PartyId)
FROM assoc
LEFT JOIN cal
  ON assoc.CALFinancialAccountNumber = cal.FinancialAccountNumber
LEFT JOIN p
  ON cal.PrimaryOwnerPartyId = p.PartyId
""")

In [19]:
# Merge to Wide Table: All 4 internal relationships
con.execute("""
CREATE OR REPLACE TABLE Internal_Wide_All AS
WITH
core AS (SELECT * FROM Broadridge_Core_All),
idv  AS (SELECT * FROM Identity_All),
act  AS (SELECT * FROM Broadridge_Activity_All),
calx AS (SELECT * FROM CAL_All)
SELECT
  core.*,

  -- Identity side (prefix)
  idv.PartyId AS id_PartyId,
  idv.* EXCLUDE (PartyId),

  -- Activity side (prefix)
  act.DeltaDate AS act_DeltaDate,
  act.* EXCLUDE (FinancialAccountId, DeltaDate),

  -- CAL side (prefix)
  calx.cal_BroadridgeFinancialAccountId,
  calx.cal_CALFinancialAccountNumber,
  calx.* EXCLUDE (cal_BroadridgeFinancialAccountId, cal_CALFinancialAccountNumber)

FROM core
LEFT JOIN idv
  ON core.bar_PartyId = idv.PartyId
LEFT JOIN act
  ON core.FinancialAccountId = act.FinancialAccountId
LEFT JOIN calx
  ON core.FinancialAccountId = calx.cal_BroadridgeFinancialAccountId
""")

In [20]:
con.execute("SELECT COUNT(*) AS n FROM Internal_Wide_All").fetchdf()
con.execute("SELECT * FROM Internal_Wide_All LIMIT 5").fetchdf()

,PrimaryOwnerPartyId,FinancialAccountId,RepCode,BranchCode,FirmId,RecordType,AccountType,AccountSubType,AccountSource,ExternalAccount,...,TaxClassification_2,TotalAssets_2,InvestedAssets_2,LiquidityNeeds_3,RiskTolerance_3,InvestmentTimeHorizon_3,InvestmentObjective_2,InvestmentExperience_2,PreferredPhoneType_2,PartyRegisteredInWMO_2
0,3281496,12433910,000N8L,010WZ,1101,Investment Asset,Cash,Individual Retirement Account (IR),Broadridge,False,...,None,None,None,None,None,None,None,None,None,<NA>
1,807431,2178348,000SCO,010SC,1101,Investment Asset,Cash,Segregated IRA Rollover (RR),Broadridge,False,...,None,None,None,None,None,None,None,None,None,<NA>
2,16359966,5510832,0003D4,010SC,1101,Investment Asset,Cash,Roth IRA or Contributory IRA (RO),Broadridge,False,...,None,None,None,None,None,None,None,None,None,<NA>
3,23277506,9295407,000SFQ,010SF,1101,Investment Asset,Cash,Individual Retirement Account (IR),Broadridge,False,...,None,None,None,None,None,None,None,None,None,<NA>
4,16681464,9235876,000SPO,010SP,1101,Investment Asset,Cash,Decedent IRA (DI),Broadridge,False,...,None,None,None,None,None,None,None,None,None,<NA>


In [21]:
import os
os.makedirs("outputs", exist_ok=True)

tables = [
    "Identity_All",
    "Broadridge_Core_All",
    "Broadridge_Activity_All",
    "CAL_All",
    "Internal_Wide_All",   # 你的最终宽表名（如果不同改这里）
]

for t in tables:
    con.execute(f"""
        COPY (SELECT * FROM "{t}")
        TO 'outputs/{t}.parquet'
        (FORMAT PARQUET);
    """)

os.listdir("outputs")

['Internal_Wide_All.parquet',
 'CAL_All.parquet',
 'Broadridge_Core_All.parquet',
 'Broadridge_Activity_All.parquet',
 'Identity_All.parquet']

In [34]:
pip install pyarrow
import pandas as pd
df = pd.read_parquet("Identity_All.parquet")
df.head()

SyntaxError: invalid syntax (2073951917.py, line 1)

In [29]:
con.execute("""
CREATE OR REPLACE TABLE HeldAway_Long AS
WITH
ha529 AS (SELECT * FROM read_csv_auto('HA529Account.csv', header=True)),
haann AS (SELECT * FROM read_csv_auto('HAAnnuityInsuranceAccount.csv', header=True)),
haret AS (SELECT * FROM read_csv_auto('HARetirementPlanAccount.csv', header=True))
SELECT
  PrimaryOwnerPartyId,
  'HA529' AS ExternalType,
  RBCHeldAccountId,
  FinancialAccountNumber,
  TRY_CAST(Balance AS DOUBLE) AS Balance,
  NULL::DOUBLE AS CashValue
FROM ha529
UNION ALL
SELECT
  PrimaryOwnerPartyId,
  'HA_ANNUITY_INSURANCE' AS ExternalType,
  RBCHeldAccountId,
  FinancialAccountNumber,
  NULL::DOUBLE AS Balance,
  TRY_CAST(CashValue AS DOUBLE) AS CashValue
FROM haann
UNION ALL
SELECT
  PrimaryOwnerPartyId,
  'HA_RETIREMENT_PLAN' AS ExternalType,
  RBCHeldAccountId,
  FinancialAccountNumber,
  TRY_CAST(Balance AS DOUBLE) AS Balance,
  NULL::DOUBLE AS CashValue
FROM haret
""")

con.execute("""
CREATE OR REPLACE TABLE HeldAway_Party AS
SELECT
  PrimaryOwnerPartyId,
  COUNT(*) AS ha_rows,
  COUNT(DISTINCT FinancialAccountNumber) AS ha_distinct_accounts,

  MAX(CASE WHEN ExternalType='HA529' THEN 1 ELSE 0 END) AS has_ha529,
  MAX(CASE WHEN ExternalType='HA_ANNUITY_INSURANCE' THEN 1 ELSE 0 END) AS has_ha_annuity_ins,
  MAX(CASE WHEN ExternalType='HA_RETIREMENT_PLAN' THEN 1 ELSE 0 END) AS has_ha_retirement,

  SUM(CASE WHEN ExternalType='HA529' THEN 1 ELSE 0 END) AS n_ha529,
  SUM(CASE WHEN ExternalType='HA_ANNUITY_INSURANCE' THEN 1 ELSE 0 END) AS n_ha_annuity_ins,
  SUM(CASE WHEN ExternalType='HA_RETIREMENT_PLAN' THEN 1 ELSE 0 END) AS n_ha_retirement,

  SUM(COALESCE(Balance,0)) AS ha_sum_balance,
  SUM(COALESCE(CashValue,0)) AS ha_sum_cash_value
FROM HeldAway_Long
GROUP BY 1
""")

In [24]:
con.execute("""
CREATE OR REPLACE TABLE Yodlee_Party AS
WITH y AS (SELECT * FROM read_csv_auto('YodleeAccount.csv', header=True))
SELECT
  PrimaryOwnerPartyId,
  COUNT(*) AS yodlee_rows,
  COUNT(DISTINCT FinancialAccountNumber) AS yodlee_distinct_accounts,

  SUM(COALESCE(TRY_CAST(CashValue AS DOUBLE), 0)) AS yodlee_sum_cash_value,
  SUM(COALESCE(TRY_CAST(OutstandingBalance AS DOUBLE), 0)) AS yodlee_sum_outstanding_balance,
  SUM(COALESCE(TRY_CAST(PortfolioBalance AS DOUBLE), 0)) AS yodlee_sum_portfolio_balance,

  MAX(AggregationStartDate) AS yodlee_latest_aggregation_start_date
FROM y
GROUP BY 1
""")

In [30]:
con.execute("""
CREATE OR REPLACE TABLE Wide_All AS
WITH
base AS (SELECT * FROM Internal_Wide_All),
ha   AS (SELECT * FROM HeldAway_Party),
yo   AS (SELECT * FROM Yodlee_Party)
SELECT
  base.*,

  ha.ha_rows,
  ha.ha_distinct_accounts,
  ha.has_ha529,
  ha.has_ha_annuity_ins,
  ha.has_ha_retirement,
  ha.n_ha529,
  ha.n_ha_annuity_ins,
  ha.n_ha_retirement,
  ha.ha_sum_balance,
  ha.ha_sum_cash_value,

  yo.yodlee_rows,
  yo.yodlee_distinct_accounts,
  yo.yodlee_sum_cash_value,
  yo.yodlee_sum_outstanding_balance,
  yo.yodlee_sum_portfolio_balance,
  yo.yodlee_latest_aggregation_start_date

FROM base
LEFT JOIN ha
  ON base.PrimaryOwnerPartyId = ha.PrimaryOwnerPartyId
LEFT JOIN yo
  ON base.PrimaryOwnerPartyId = yo.PrimaryOwnerPartyId
""")

In [31]:
con.execute("SELECT COUNT(*) AS n FROM Wide_All").fetchdf()
con.execute("SELECT * FROM Wide_All LIMIT 5").fetchdf()

,PrimaryOwnerPartyId,FinancialAccountId,RepCode,BranchCode,FirmId,RecordType,AccountType,AccountSubType,AccountSource,ExternalAccount,...,n_ha_annuity_ins,n_ha_retirement,ha_sum_balance,ha_sum_cash_value,yodlee_rows,yodlee_distinct_accounts,yodlee_sum_cash_value,yodlee_sum_outstanding_balance,yodlee_sum_portfolio_balance,yodlee_latest_aggregation_start_date
0,3018118,771953,000TSJ,010DH,1101,Investment Asset,Cash,Individual Retirement Account (IR),Broadridge,False,...,1.0,0.0,0.00000,58380.13,<NA>,<NA>,NaN,NaN,NaN,NaT
1,3400195,944077,0009U7,010SF,1101,Investment Asset,Cash,Roth IRA or Contributory IRA (RO),Broadridge,False,...,1.0,0.0,8385.46932,34366.33,<NA>,<NA>,NaN,NaN,NaN,NaT
2,3117258,163429,000GGV,010DH,1101,Investment Asset,Cash,Individual Retirement Account (IR),Broadridge,False,...,2.0,0.0,0.00000,462168.60,<NA>,<NA>,NaN,NaN,NaN,NaT
3,259741,2441558,000GGV,010DH,1101,Investment Asset,Cash,Individual Retirement Account (IR),Broadridge,False,...,3.0,0.0,0.00000,422826.13,<NA>,<NA>,NaN,NaN,NaN,NaT
4,22162758,8583623,0008PW,010FA,1101,Investment Asset,Cash,Individual Retirement Account (IR),Broadridge,False,...,2.0,0.0,0.00000,271215.69,<NA>,<NA>,NaN,NaN,NaN,NaT


In [32]:
import os
os.makedirs("outputs", exist_ok=True)

for t in ["HeldAway_Long","HeldAway_Party","Yodlee_Party","Wide_All"]:
    con.execute(f"""
        COPY (SELECT * FROM "{t}")
        TO 'outputs/{t}.parquet'
        (FORMAT PARQUET);
    """)

In [39]:
import pandas as pd
df = pd.read_parquet("outputs/Yodlee_Party.parquet")
df

,PrimaryOwnerPartyId,yodlee_rows,yodlee_distinct_accounts,yodlee_sum_cash_value,yodlee_sum_outstanding_balance,yodlee_sum_portfolio_balance,yodlee_latest_aggregation_start_date
0,29999130,24,24,0.0,0.0,1698672.01,2025-02-07
1,3262997,3,3,0.0,0.0,625797.68,2024-05-01
2,354642,4,2,0.0,0.0,33315459.62,2024-04-30
3,1158126,20,10,0.0,0.0,2845449.42,2024-05-01
4,5575883,7,7,0.0,0.0,5053114.54,2024-05-24
...,...,...,...,...,...,...,...
990,22166566,1,1,0.0,0.0,58546.81,2024-05-02
991,4506162,4,4,0.0,0.0,1789660.10,2024-04-29
992,28892035,1,1,0.0,0.0,427826.46,2025-02-27
993,19542885,1,1,0.0,0.0,267721.11,2025-10-21


In [40]:
import duckdb
import pandas as pd

# Option A (recommended for big files): DuckDB reads Parquet and returns first N rows
con = duckdb.connect()

files = [
    "Broadridge_Activity_All.parquet",
    "Broadridge_Core_All.parquet",
    "CAL_All.parquet",
    "HeldAway_Long.parquet",
    "HeldAway_Party.parquet",
    "Identity_All.parquet",
    "Internal_Wide_All.parquet",
    "Wide_All.parquet",
    "Yodlee_Party.parquet",
]

N = 5  # change to 10 if you want

for f in files:
    path = f"outputs/{f}"
    print(f"\n=== {f} (first {N} rows) ===")
    df = con.execute(f"SELECT * FROM read_parquet('{path}') LIMIT {N}").fetchdf()
    display(df)


=== Broadridge_Activity_All.parquet (first 5 rows) ===


,FinancialAccountId,DeltaDate,PrimaryOwnerPartyId,ActionCD,PYTDIRAContribution,PYinCYIRAContribution,CYTDIRAContribution,PYTDEmployeeContribution,PYinCYEmployeeContribution,CYTDEmployeeContribution,...,PYTDSimpleRollover,CYTDSimpleRollover,PrimaryOwnerPartyId_1,ActionCD_1,PYTDGrossDistribution,CYTDGrossDistribution,PYTDFedWithholding,CYTDFedWithholding,PYTDStateWithholding,CYTDStateWithholding
0,789634,2024-04-18,3249059,A,NULL,NULL,NULL,NULL,NULL,NULL,...,NULL,NULL,3249059,A,1976.00,NULL,395.20,NULL,296.40,NULL
1,789640,2024-04-18,3249086,A,NULL,NULL,NULL,NULL,NULL,NULL,...,NULL,NULL,3249086,A,17385.00,NULL,3477.00,NULL,2607.75,NULL
2,6661,2024-04-18,325024,A,NULL,NULL,NULL,NULL,NULL,NULL,...,NULL,NULL,325024,A,NULL,NULL,NULL,NULL,NULL,NULL
3,880194,2024-04-18,3250280,A,NULL,NULL,NULL,NULL,NULL,NULL,...,NULL,NULL,3250280,A,28250.00,4000.00,2825.00,400.00,2825.00,400.00
4,4330247,2024-04-18,3250330,A,NULL,NULL,NULL,NULL,NULL,NULL,...,NULL,NULL,3250330,A,NULL,NULL,NULL,NULL,NULL,NULL



=== Broadridge_Core_All.parquet (first 5 rows) ===


,PrimaryOwnerPartyId,FinancialAccountId,RepCode,BranchCode,FirmId,RecordType,AccountType,AccountSubType,AccountSource,ExternalAccount,...,TaxClassification,TotalAssets,InvestedAssets,LiquidityNeeds_1,RiskTolerance_1,InvestmentTimeHorizon_1,InvestmentObjective,InvestmentExperience,PreferredPhoneType,PartyRegisteredInWMO
0,15837687,5190604,000SCO,010SC,1101,Investment Asset,Cash,Decedent IRA (DI),Broadridge,False,...,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,False
1,4227534,6690384,000SPO,010SP,1101,Investment Asset,Cash,Decedent IRA (DI),Broadridge,False,...,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,False
2,6270532,2395213,000V98,010SP,1101,Investment Asset,Margin,Individual Account,Broadridge,False,...,NULL,NULL,NULL,NULL,NULL,NULL,NULL,Bonds;Stocks,Home Phone,False
3,6010554,778784,000RIO,010RI,1101,Investment Asset,Cash,Individual Retirement Account (IR),Broadridge,False,...,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,False
4,16615,1236398,0002VL,010RI,1101,Investment Asset,Held Away Account Only,Held Away 529 College Savings Plan,Broadridge,False,...,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,False



=== CAL_All.parquet (first 5 rows) ===


...

,PrimaryOwnerPartyId,FinancialAccountId,RepCode,BranchCode,FirmId,RecordType,AccountType,AccountSubType,AccountSource,ExternalAccount,...,TaxClassification_2,TotalAssets_2,InvestedAssets_2,LiquidityNeeds_3,RiskTolerance_3,InvestmentTimeHorizon_3,InvestmentObjective_2,InvestmentExperience_2,PreferredPhoneType_2,PartyRegisteredInWMO_2
0,3281496,12433910,000N8L,010WZ,1101,Investment Asset,Cash,Individual Retirement Account (IR),Broadridge,False,...,None,None,None,None,None,None,None,None,None,<NA>
1,807431,2178348,000SCO,010SC,1101,Investment Asset,Cash,Segregated IRA Rollover (RR),Broadridge,False,...,None,None,None,None,None,None,None,None,None,<NA>
2,16359966,5510832,0003D4,010SC,1101,Investment Asset,Cash,Roth IRA or Contributory IRA (RO),Broadridge,False,...,None,None,None,None,None,None,None,None,None,<NA>
3,23277506,9295407,000SFQ,010SF,1101,Investment Asset,Cash,Individual Retirement Account (IR),Broadridge,False,...,None,None,None,None,None,None,None,None,None,<NA>
4,16681464,9235876,000SPO,010SP,1101,Investment Asset,Cash,Decedent IRA (DI),Broadridge,False,...,None,None,None,None,None,None,None,None,None,<NA>



=== Wide_All.parquet (first 5 rows) ===


,PrimaryOwnerPartyId,FinancialAccountId,RepCode,BranchCode,FirmId,RecordType,AccountType,AccountSubType,AccountSource,ExternalAccount,...,n_ha_annuity_ins,n_ha_retirement,ha_sum_balance,ha_sum_cash_value,yodlee_rows,yodlee_distinct_accounts,yodlee_sum_cash_value,yodlee_sum_outstanding_balance,yodlee_sum_portfolio_balance,yodlee_latest_aggregation_start_date
0,3018118,771953,000TSJ,010DH,1101,Investment Asset,Cash,Individual Retirement Account (IR),Broadridge,False,...,1.0,0.0,0.00000,58380.13,<NA>,<NA>,NaN,NaN,NaN,NaT
1,3400195,944077,0009U7,010SF,1101,Investment Asset,Cash,Roth IRA or Contributory IRA (RO),Broadridge,False,...,1.0,0.0,8385.46932,34366.33,<NA>,<NA>,NaN,NaN,NaN,NaT
2,3117258,163429,000GGV,010DH,1101,Investment Asset,Cash,Individual Retirement Account (IR),Broadridge,False,...,2.0,0.0,0.00000,462168.60,<NA>,<NA>,NaN,NaN,NaN,NaT
3,259741,2441558,000GGV,010DH,1101,Investment Asset,Cash,Individual Retirement Account (IR),Broadridge,False,...,3.0,0.0,0.00000,422826.13,<NA>,<NA>,NaN,NaN,NaN,NaT
4,22162758,8583623,0008PW,010FA,1101,Investment Asset,Cash,Individual Retirement Account (IR),Broadridge,False,...,2.0,0.0,0.00000,271215.69,<NA>,<NA>,NaN,NaN,NaN,NaT



=== Yodlee_Party.parquet (first 5 rows) ===


,PrimaryOwnerPartyId,yodlee_rows,yodlee_distinct_accounts,yodlee_sum_cash_value,yodlee_sum_outstanding_balance,yodlee_sum_portfolio_balance,yodlee_latest_aggregation_start_date
0,29999130,24,24,0.0,0.0,1698672.01,2025-02-07
1,3262997,3,3,0.0,0.0,625797.68,2024-05-01
2,354642,4,2,0.0,0.0,33315459.62,2024-04-30
3,1158126,20,10,0.0,0.0,2845449.42,2024-05-01
4,5575883,7,7,0.0,0.0,5053114.54,2024-05-24
